# Sample and verify the results of the deduplication

In [13]:
from pathlib import Path
from IPython.display import Markdown, display
import polars as pl

ENTITY_CLUSTERS = Path("data/entity_clusters.parquet")
CLASSIFIED_GRANTS = Path("data/classified_grants.parquet")

display(Markdown("Dependencies loaded and input paths configured."))

Dependencies loaded and input paths configured.

In [14]:
clusters_exist = ENTITY_CLUSTERS.exists()
classified_exist = CLASSIFIED_GRANTS.exists()

display(Markdown(f"""
## Input Files

- `entity_clusters.parquet`: {'found' if clusters_exist else 'missing'}
- `classified_grants.parquet`: {'found' if classified_exist else 'missing'}
"""))


## Input Files

- `entity_clusters.parquet`: found
- `classified_grants.parquet`: found


In [15]:
if not clusters_exist or not classified_exist:
    ec = None
    display(Markdown("Required files missing. Run notebooks 03 and 05 first."))
else:
    ec = pl.read_parquet(ENTITY_CLUSTERS)
    cg = pl.read_parquet(CLASSIFIED_GRANTS)

    join_cols = [
        "recipient_legal_name_en",
        "recipient_type",
        "recipient_business_number",
        "recipient_postal_code",
        "recipient_city",
        "owner_org",
    ]
    available_join = [col for col in join_cols if col in cg.columns and col not in ec.columns]
    if available_join:
        ec = ec.join(
            cg.select(["unique_id"] + available_join) if "unique_id" in cg.columns else cg.select(available_join),
            on="unique_id",
            how="left",
        )

    display(Markdown(f"""
## Data Loaded

- **Entity clusters:** {ec.height:,} rows
- **Columns:** {len(ec.columns)}
"""))


## Data Loaded

- **Entity clusters:** 1,303,723 rows
- **Columns:** 54


In [16]:
if ec is None:
    cluster_sizes = None
    total_rows = None
    n_clusters = None
    display(Markdown("No data loaded."))
else:
    total_rows = ec.height
    n_clusters = ec["cluster_id"].n_unique()
    avg_size = round(total_rows / n_clusters, 1) if n_clusters > 0 else 0
    cluster_sizes = ec.group_by("cluster_id").agg(pl.len().alias("size"))
    median_size = cluster_sizes["size"].median()
    max_size = cluster_sizes["size"].max()
    singletons = cluster_sizes.filter(pl.col("size") == 1).height
    singleton_pct = singletons / n_clusters * 100 if n_clusters else 0
    reduction_pct = (1 - n_clusters / total_rows) * 100 if total_rows else 0

    display(Markdown(f"""
## Overview

| Metric | Value |
|--------|-------|
| Total grant rows | {total_rows:,} |
| Unique entities (clusters) | {n_clusters:,} |
| Avg grants per entity | {avg_size} |
| Median grants per entity | {median_size} |
| Max grants per entity | {max_size:,} |
| Singletons (1 grant) | {singletons:,} ({singleton_pct:.1f}%) |
| Reduction ratio | {reduction_pct:.1f}% |
"""))


## Overview

| Metric | Value |
|--------|-------|
| Total grant rows | 1,303,723 |
| Unique entities (clusters) | 353,516 |
| Avg grants per entity | 3.7 |
| Median grants per entity | 1.0 |
| Max grants per entity | 4,033 |
| Singletons (1 grant) | 201,643 (57.0%) |
| Reduction ratio | 72.9% |


In [17]:
if cluster_sizes is None:
    display(Markdown("No data loaded."))
else:
    size_dist = (
        cluster_sizes
        .with_columns(
            pl.when(pl.col("size") == 1).then(pl.lit("1 (singleton)"))
            .when(pl.col("size") == 2).then(pl.lit("2"))
            .when(pl.col("size").is_between(3, 5)).then(pl.lit("3-5"))
            .when(pl.col("size").is_between(6, 10)).then(pl.lit("6-10"))
            .when(pl.col("size").is_between(11, 20)).then(pl.lit("11-20"))
            .when(pl.col("size").is_between(21, 50)).then(pl.lit("21-50"))
            .when(pl.col("size").is_between(51, 100)).then(pl.lit("51-100"))
            .otherwise(pl.lit("100+"))
            .alias("size_bucket")
        )
        .group_by("size_bucket")
        .agg(pl.len().alias("num_clusters"), pl.col("size").sum().alias("total_rows"))
        .with_columns((pl.col("total_rows") / pl.col("total_rows").sum() * 100).round(1).alias("pct_of_rows"))
        .sort("size_bucket")
    )
    display(Markdown("### Cluster Size Distribution"))
    display(size_dist)

### Cluster Size Distribution

size_bucket,num_clusters,total_rows,pct_of_rows
str,u32,u32,f64
"""1 (singleton)""",201643,201643,15.5
"""100+""",977,282387,21.7
"""11-20""",10094,141903,10.9
"""2""",54656,109312,8.4
"""21-50""",3772,111422,8.5
"""3-5""",54802,203731,15.6
"""51-100""",743,51717,4.0
"""6-10""",26829,201608,15.5


In [18]:
if cluster_sizes is None:
    display(Markdown("No data loaded."))
else:
    top_20 = cluster_sizes.sort("size", descending=True).head(20)
    top_ids = top_20["cluster_id"].to_list()
    sample_cols = [col for col in [
        "cluster_id", "unique_id", "recipient_legal_name_en", "recipient_type",
        "recipient_business_number", "recipient_city", "recipient_postal_code",
    ] if col in ec.columns]
    top_detail = (
        ec.filter(pl.col("cluster_id").is_in(top_ids))
        .select(sample_cols)
        .sort(["cluster_id", "unique_id"])
    )
    display(Markdown("### Largest 20 Clusters"))
    display(top_20)
    display(Markdown("### Detail of Largest Clusters (spot-check for over-linking)"))
    display(top_detail)

### Largest 20 Clusters

cluster_id,size
u32,u32
3623283,4033
3621166,2958
5210580,2643
3517804,2208
3621201,2098
…,…
3621148,1706
3622203,1456
3621081,1420


### Detail of Largest Clusters (spot-check for over-linking)

cluster_id,unique_id,recipient_legal_name_en,recipient_type,recipient_business_number,recipient_city,recipient_postal_code
u32,u32,str,str,str,str,str
3517804,911796,"""jonathan paquin""","""P""","""119278950""","""QUEBEC""","""G1V0A6"""
3517804,1015074,"""lalonde, jean-françois""","""P""",null,"""QUEBEC""","""G1V0A6"""
3517804,1015189,"""gaudreault, jonathan""","""P""",null,"""QUEBEC""","""G1V0A6"""
3517804,1015190,"""kaliaguine, serge""","""P""",null,"""QUEBEC""","""G1V0A6"""
3517804,1015589,"""lebel, luc""","""P""",null,"""QUEBEC""","""G1V0A6"""
…,…,…,…,…,…,…
5213412,1291460,"""the governors of the universit…","""S""","""108102831""","""Edmonton""","""T6G2E1"""
5213412,1291463,"""the governors of the universit…","""S""","""108102831""","""Edmonton""","""T6G2E1"""
5213412,1295928,"""the governors of the universit…","""S""","""108102831""","""Edmonton""","""T5J4P6"""


In [19]:
if ec is None or "recipient_type" not in ec.columns:
    cross_type = None
    display(Markdown("No `recipient_type` column available for cross-type check."))
else:
    cross_type = (
        ec.group_by("cluster_id")
        .agg(pl.col("recipient_type").n_unique().alias("n_types"), pl.len().alias("size"))
        .filter(pl.col("n_types") > 1)
        .sort("n_types", descending=True)
    )
    n_inconsistent = cross_type.height
    pct_inconsistent = round(n_inconsistent / ec["cluster_id"].n_unique() * 100, 1)
    display(Markdown(f"""
### Cross-Type Consistency

Entities with grants tagged as **different recipient types**: **{n_inconsistent:,}** ({pct_inconsistent}% of entities)
"""))
    display(cross_type.head(30))


### Cross-Type Consistency

Entities with grants tagged as **different recipient types**: **0** (0.0% of entities)


cluster_id,n_types,size
u32,u32,u32


In [20]:
if cross_type is None or cross_type.height == 0:
    display(Markdown("No cross-type inconsistencies found or no data."))
else:
    worst_ids = cross_type.head(10)["cluster_id"].to_list()
    detail_cols = [col for col in [
        "cluster_id", "unique_id", "recipient_legal_name_en", "recipient_type",
        "recipient_business_number", "recipient_city",
    ] if col in ec.columns]
    worst_detail = (
        ec.filter(pl.col("cluster_id").is_in(worst_ids))
        .select(detail_cols)
        .sort(["cluster_id", "recipient_type"])
    )
    display(Markdown("### Most Inconsistent Clusters (detail)"))
    display(worst_detail)

No cross-type inconsistencies found or no data.

In [21]:
if ec is None or "recipient_type" not in ec.columns:
    display(Markdown("No `recipient_type` column for before/after comparison."))
else:
    before_after = (
        ec.group_by("recipient_type")
        .agg(
            pl.len().alias("original_rows"),
            pl.col("cluster_id").n_unique().alias("entities"),
        )
        .with_columns(
            (pl.col("original_rows") - pl.col("entities")).alias("rows_eliminated"),
            ((1 - pl.col("entities") / pl.col("original_rows")) * 100).round(1).alias("reduction_pct"),
        )
        .sort("original_rows", descending=True)
    )
    display(Markdown("### Before vs After by Recipient Type"))
    display(before_after)

### Before vs After by Recipient Type

recipient_type,original_rows,entities,rows_eliminated,reduction_pct
str,u32,u32,u32,f64
"""F""",357071,128795,228276,63.9
"""N""",343124,75750,267374,77.9
"""P""",246573,90865,155708,63.1
"""A""",176084,6889,169195,96.1
"""S""",64065,20777,43288,67.6
"""O""",60425,15714,44711,74.0
"""G""",52044,11295,40749,78.3
"""I""",4337,3431,906,20.9


In [22]:
SEARCH_QUERY = ""  # Set this to an entity name fragment, e.g. "Dalhousie"

query = SEARCH_QUERY.strip()
if not query or ec is None:
    display(Markdown("Set `SEARCH_QUERY` above and rerun this cell to search entity names."))
else:
    name_cols = [col for col in [
        "recipient_legal_name_en", "recipient_operating_name_en",
        "research_organization_name_en",
    ] if col in ec.columns]
    if not name_cols:
        display(Markdown("No name columns found."))
    else:
        or_expr = pl.lit(False)
        for col in name_cols:
            or_expr = or_expr | pl.col(col).str.contains(f"(?i){query}")
        matches = ec.filter(or_expr)
        if matches.height == 0:
            display(Markdown(f"No matches for **{query}**."))
        else:
            matched_cluster_ids = matches["cluster_id"].unique().to_list()
            full_entity = ec.filter(pl.col("cluster_id").is_in(matched_cluster_ids))
            search_cols = [col for col in [
                "cluster_id", "unique_id", "recipient_legal_name_en", "recipient_type",
                "recipient_type_source", "recipient_business_number", "recipient_city",
                "recipient_postal_code", "owner_org",
            ] if col in ec.columns]
            display(Markdown(f"**{matches.height:,}** rows match `{query}` across **{len(matched_cluster_ids):,}** entities."))
            display(full_entity.select(search_cols).sort(["cluster_id", "unique_id"]))

Set `SEARCH_QUERY` above and rerun this cell to search entity names.

In [23]:
RANDOM_SEED = 42
SAMPLE_CLUSTERS = 10

if ec is None:
    display(Markdown("No data loaded."))
else:
    all_ids = ec["cluster_id"].unique().sort()
    k = min(int(SAMPLE_CLUSTERS), all_ids.len())
    sampled = all_ids.sample(n=k, seed=int(RANDOM_SEED)).sort()
    show_cols = [col for col in [
        "cluster_id", "unique_id", "recipient_legal_name_en", "recipient_type",
        "recipient_business_number", "recipient_city", "recipient_postal_code",
    ] if col in ec.columns]
    sample_detail = (
        ec.filter(pl.col("cluster_id").is_in(sampled.to_list()))
        .select(show_cols)
        .sort(["cluster_id", "unique_id"])
    )
    display(Markdown(f"### Random Cluster Sample ({k} clusters, seed {RANDOM_SEED})"))
    display(sample_detail)

### Random Cluster Sample (10 clusters, seed 42)

cluster_id,unique_id,recipient_legal_name_en,recipient_type,recipient_business_number,recipient_city,recipient_postal_code
u32,u32,str,str,str,str,str
402295,402295,"""advanced lighting systems inc""","""F""","""805767472""","""Concord""","""L4K1W6"""
402295,427238,"""advanced lighting systems inc""","""F""","""805767472""","""Concord""","""L4K1W6"""
402295,450664,"""advanced lighting systems inc""","""F""","""805767472""","""Concord""","""L4K1W6"""
874556,874556,"""2582606 ontario ltd""","""F""","""709536494""","""THORNHILL""",null
953435,953435,"""10493279 canada inc""","""F""","""784721318""","""Montreal|Montréal""","""H4A2R2"""
…,…,…,…,…,…,…
6835436,322677,"""gwichin tribal council""","""O""",null,"""Inuvik""",null
6835436,323270,"""gwichin tribal council""","""O""",null,"""Inuvik""",null
8523335,707745,"""region of peel""","""G""",null,"""Peel""","""L6T4B9"""


In [24]:
if ec is None:
    display(Markdown("No data loaded."))
elif "owner_org" not in ec.columns:
    display(Markdown("No `owner_org` column available."))
else:
    by_owner = (
        ec.group_by("owner_org")
        .agg(
            pl.len().alias("total_rows"),
            pl.col("cluster_id").n_unique().alias("entities"),
        )
        .with_columns(
            (pl.col("total_rows") - pl.col("entities")).alias("rows_eliminated"),
            ((1 - pl.col("entities") / pl.col("total_rows")) * 100).round(1).alias("reduction_pct"),
        )
        .sort("total_rows", descending=True)
        .head(30)
    )
    display(Markdown("### Deduplication by Owner Organization (top 30)"))
    display(by_owner)

### Deduplication by Owner Organization (top 30)

owner_org,total_rows,entities,rows_eliminated,reduction_pct
str,u32,u32,u32,f64
"""esdc-edsc""",352754,92055,260699,73.9
"""isc-sac""",156896,2536,154360,98.4
"""nserc-crsng""",151553,37745,113808,75.1
"""pch""",134219,58923,75296,56.1
"""nrc-cnrc""",77946,26267,51679,66.3
…,…,…,…,…
"""prairiescan""",3396,1904,1492,43.9
"""pc""",3024,968,2056,68.0
"""wage""",2578,1330,1248,48.4
